# Fine-Tuning Large Language Models — Hands-On Notebook (LoRA / QLoRA)

Companion notebook to `Fine_Tuning_LLMs_Tutorial.md`. Section numbers below refer to that document.

**Runs on a single free Google Colab T4 GPU (16 GB).** No paid compute, no API keys, no private data.

*Runtime → Change runtime type → T4 GPU*, then run the cells in order.

| Step | Approx. time |
|---|---|
| Install libraries | 2–5 min |
| Download + quantize the 1.5B model | 1–3 min |
| Train (2,000 examples, 1 epoch) | 10–25 min |
| Evaluate + generate | 3–5 min |

If you are short on time, change the dataset slice in Cell 4 from `train[:2000]` to `train[:500]` — every concept still applies.

## §7.3 — Cell 1: install

Pinning majors keeps this notebook reproducible. Remove the pins to track latest.

In [ ]:
# Cell 1 — install (Colab). Takes ~2-5 minutes.
# Pinning majors keeps this notebook reproducible. Remove the pins to track latest.
!pip install -q unsloth
!pip install -q "trl>=1.0,<2.0" "transformers>=5.0,<6.0" peft accelerate bitsandbytes datasets

# If the install misbehaves on Colab, the dev build is often ahead of PyPI:
# !pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## §7.4 — Cell 2: environment check

Always run this. It takes two seconds and saves an hour.

> **Precision trap.** A T4 is Turing architecture and does **not** support bfloat16. Hard-coding `bf16=True` will
> fail or silently degrade on the free tier. We compute `BF16` once here and use it everywhere.

In [ ]:
# Cell 2 — verify GPU and record the exact environment
import torch, platform

assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"

gpu   = torch.cuda.get_device_name(0)
vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16  = torch.cuda.is_bf16_supported()   # False on T4 (Turing); True on Ampere+ (A100, L4, ...)

print(f"GPU            : {gpu} ({vram:.1f} GB)")
print(f"bfloat16 support: {BF16}   -> we will train in {'bf16' if BF16 else 'fp16'}")
print(f"Python         : {platform.python_version()} | torch {torch.__version__}")

import trl, transformers, peft
print(f"trl {trl.__version__} | transformers {transformers.__version__} | peft {peft.__version__}")

## §8.1 — Cell 3: load the base model in 4-bit and attach LoRA adapters

This is the "QLoRA" setup: a frozen 4-bit base model plus trainable LoRA adapters.

Compare the printed trainable-parameter count against the hand calculation in §6.3 (≈ 18.5 M, about 1.2 % of 1.54 B).

In [ ]:
# Cell 3 — this is the "QLoRA" setup: 4-bit frozen base + trainable LoRA adapters
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
SEED        = 3407

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Qwen2.5-1.5B-Instruct",
    # Faster download (already quantized): "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
    # Scale up later with:                 "unsloth/Qwen2.5-7B-Instruct"
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,   # <- the "Q" in QLoRA
    dtype          = None,   # None = auto: fp16 on T4/V100, bf16 on Ampere+
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,            # LoRA rank -> capacity (see 6.3 for the param count)
    lora_alpha     = 32,            # update scale; ~2x rank is a solid default
    lora_dropout   = 0.05,          # light regularization; 0 is also common
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",   # attention
                      "gate_proj", "up_proj", "down_proj"],     # MLP
    bias           = "none",
    use_gradient_checkpointing = "unsloth",   # big activation-memory saver
    use_rslora     = False,         # set True if you raise r to 64+
    random_state   = SEED,
)

model.print_trainable_parameters()   # compare this against your hand calculation from 6.3

## §8.2 — Cell 4: load, format, and split the dataset

Instruction models are trained with a **chat template** — special tokens marking who is speaking. Always format your
training data with the tokenizer's *own* template, and prompt the same way at inference. A mismatch here degrades
quality quietly, with no error message.

**Read the printed example carefully.** You should see the control tokens (`<|im_start|>user`, `<|im_end|>`,
`<|im_start|>assistant`) wrapping the content. Raw text with no special tokens means the template was not applied.

In [ ]:
# Cell 4 — load a public instruction dataset, apply the chat template, hold out a validation split
from datasets import load_dataset

raw = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")   # 2k is plenty for a demo

def format_example(ex):
    user = ex["instruction"] if not ex["input"] else f'{ex["instruction"]}\n\n{ex["input"]}'
    messages = [
        {"role": "user",      "content": user},
        {"role": "assistant", "content": ex["output"]},
    ]
    # add_generation_prompt=False -> we want the completed assistant turn, for training
    ex["text"] = tokenizer.apply_chat_template(messages, tokenize=False,
                                               add_generation_prompt=False)
    return ex

ds = raw.map(format_example, remove_columns=raw.column_names)

# Hold out 15% so we have an honest signal (see section 10)
split      = ds.train_test_split(test_size=0.15, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]

print(f"train {len(train_ds)} | eval {len(eval_ds)}")
print("-" * 60)
print(train_ds[0]["text"][:600])   # ALWAYS eyeball one formatted example

## §8.3 — Cell 5: compatibility shim

Ten lines that make the notebook survive TRL's next rename. See Appendix B (§17) for the renames this covers.

In [ ]:
# Cell 5 — detect the installed TRL API instead of assuming it
import inspect, trl
from dataclasses import fields
from trl import SFTConfig, SFTTrainer
from packaging.version import parse as V

_CFG_FIELDS      = {f.name for f in fields(SFTConfig)}
_TRAINER_PARAMS  = set(inspect.signature(SFTTrainer.__init__).parameters)
_NEW_TO_OLD      = {"max_length": "max_seq_length"}   # renamed in TRL 0.20

def make_sft_config(**kw):
    """Build an SFTConfig using modern names, downgrading them if TRL is older."""
    out = {}
    for k, v in kw.items():
        if k in _CFG_FIELDS:
            out[k] = v
        elif k in _NEW_TO_OLD and _NEW_TO_OLD[k] in _CFG_FIELDS:
            out[_NEW_TO_OLD[k]] = v
        else:
            print(f"[compat] SFTConfig does not accept '{k}' in trl {trl.__version__} — dropping it")
    return SFTConfig(**out)

if "processing_class" in _TRAINER_PARAMS:
    TOKENIZER_KW = "processing_class"
elif "tokenizer" in _TRAINER_PARAMS:
    TOKENIZER_KW = "tokenizer"
else:                                     # signature hidden behind **kwargs
    TOKENIZER_KW = "processing_class" if V(trl.__version__) >= V("0.16") else "tokenizer"

print(f"[compat] trl {trl.__version__}: passing the tokenizer as '{TOKENIZER_KW}'")

## §8.4 — Cell 6: configure and run training

Training loss prints every 10 steps, validation loss every 50. §9 explains every argument; §10 explains how to read
the curves.

> **Note on `load_best_model_at_end`.** This restores the checkpoint with the lowest *validation* loss rather than
> leaving you with the final step's weights. On a small dataset the last checkpoint is frequently not the best one.

In [ ]:
# Cell 6 — supervised fine-tuning with TRL's SFTTrainer
cfg = make_sft_config(
    output_dir                  = "outputs",
    dataset_text_field          = "text",
    max_length                  = MAX_SEQ_LEN,   # 'max_seq_length' on TRL < 0.20 (shim handles it)

    per_device_train_batch_size = 2,     # rows per GPU step
    gradient_accumulation_steps = 4,     # -> effective batch = 2 x 4 = 8
    num_train_epochs            = 1,     # 1 epoch is enough for a demo
    learning_rate               = 2e-4,  # typical for LoRA/QLoRA
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.03,
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    optim                       = "paged_adamw_8bit",

    fp16 = not BF16,     # T4 -> fp16
    bf16 = BF16,         # Ampere+ -> bf16

    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = 50,
    save_strategy               = "steps",
    save_steps                  = 50,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    save_total_limit            = 2,

    packing                     = False,  # keep False for instruction SFT
    seed                        = SEED,
    report_to                   = "none", # set "wandb"/"tensorboard" to log properly
)

trainer = SFTTrainer(
    model         = model,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args          = cfg,
    **{TOKENIZER_KW: tokenizer},
)

trainer_stats = trainer.train()

## §8.5 — Cell 7: training report

Compare the peak VRAM figure against your prediction from §6.2. Forecasting this within a gigabyte or so is a
practical skill.

In [ ]:
# Cell 7 — record what actually happened (useful evidence for your write-up)
import math

peak_gb = torch.cuda.max_memory_reserved() / 1e9
runtime = trainer_stats.metrics["train_runtime"]
final   = trainer.evaluate()

print(f"Runtime        : {runtime/60:.1f} min")
print(f"Peak VRAM      : {peak_gb:.2f} GB of {vram:.1f} GB")
print(f"Final eval loss: {final['eval_loss']:.4f}")
print(f"Perplexity     : {math.exp(final['eval_loss']):.2f}")

### Optional: plot the loss curves (§10)

Not in the written document, but the log history is already in memory and the plot is the single most useful piece of
evidence for a write-up. Training loss should fall smoothly; validation loss is the honest signal.

In [ ]:
# Optional — plot training vs. validation loss from the trainer's log history
import matplotlib.pyplot as plt

hist  = trainer.state.log_history
train = [(h["step"], h["loss"])      for h in hist if "loss" in h]
val   = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]

plt.figure(figsize=(7, 4))
if train:
    plt.plot(*zip(*train), label="training loss")
if val:
    plt.plot(*zip(*val), marker="o", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Loss curves")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## §8.6 — Optional upgrade: train on responses only

Cell 6 computes loss over the **whole** formatted sequence, including the user's prompt — which spends capacity
teaching the model to generate *questions*. Masking the prompt tokens fixes that.

Expect the reported loss to **jump** when you enable this — that is correct, not a bug. You are averaging loss over a
smaller, harder set of tokens, so the number is not comparable to the previous run (§10.3).

This cell is optional; run Cell 6 again after it to retrain with the mask.

In [ ]:
# Option A — Unsloth helper (works with Qwen2.5 / ChatML templates)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

# Option B — TRL native (requires a conversational 'messages' column instead of 'text',
# and a chat template containing {% generation %} markers; TRL patches some model
# families automatically). Set assistant_only_loss=True in SFTConfig.

# trainer_stats = trainer.train()   # uncomment to retrain with the prompt masked

## §11.1 — Cell 8: generation helper

> Use **greedy decoding** for any before/after comparison. Sampling adds variance that will swamp the effect you are
> trying to measure.

In [ ]:
# Cell 8 — inference helper
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(model)   # ~2x faster generation

@torch.no_grad()
def generate(prompt, max_new_tokens=192, greedy=True):
    msgs = [{"role": "user", "content": prompt}]
    ids  = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        input_ids      = ids,
        attention_mask = torch.ones_like(ids),
        max_new_tokens = max_new_tokens,
        do_sample      = not greedy,          # greedy => reproducible comparisons
        temperature    = None if greedy else 0.7,
        top_p          = None if greedy else 0.9,
        pad_token_id   = tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print(generate("Explain LoRA to a beginner in two sentences."))

## §11.2 — Cell 9: the comparison that matters, base vs. fine-tuned

Because LoRA leaves the base weights untouched, you can toggle the adapter off and get the original model back from
the same memory — no second model load required.

*If `disable_adapter()` raises an error in your version, load the base model separately in a fresh runtime and save
both sets of outputs to compare offline.*

**Score these by hand** using the rubric table in §11.2. Note that prompt 4 deliberately tests a **fact**, not a
behavior — fine-tuning is not a knowledge-injection tool (§3).

In [ ]:
# Cell 9 — same weights, adapter on vs. off
PROMPTS = [
    "Give three tips for writing clear code.",
    "Summarize the water cycle in two sentences.",
    "Convert this to a bulleted list: we need milk, eggs, and bread.",
    "What is the capital of Australia?",
    "Write a polite email declining a meeting invitation.",
]

for p in PROMPTS:
    with model.disable_adapter():          # <- base model behavior
        before = generate(p)
    after = generate(p)                    # <- fine-tuned behavior
    print("=" * 78)
    print("PROMPT :", p)
    print("\n[BASE]\n", before)
    print("\n[TUNED]\n", after)

## §11.3 — Cell 10: an automatic metric

> **Caution on n-gram metrics.** ROUGE rewards surface overlap with one reference answer. For open-ended instruction
> following there are many good answers, so a low ROUGE score does not necessarily mean a bad model. Use it as a
> *relative* signal between your own runs.

*(For a clean experiment, draw these examples from `eval_ds` — data the model never trained on — rather than from the
head of the raw dataset.)*

In [ ]:
# Cell 10 — ROUGE against held-out references
!pip install -q evaluate rouge_score
import evaluate

n     = 25
refs  = [raw[i]["output"] for i in range(n)]
prompts = [
    raw[i]["instruction"] if not raw[i]["input"]
    else f'{raw[i]["instruction"]}\n\n{raw[i]["input"]}'
    for i in range(n)
]
preds = [generate(p, max_new_tokens=256) for p in prompts]

rouge = evaluate.load("rouge")
print(rouge.compute(predictions=preds, references=refs))

## §12 — Cell 11: saving, merging, and inference

| Format | Size | Best for |
|---|---|---|
| **Adapter only** | tens of MB | Sharing, versioning, serving many tasks from one base model |
| **Merged 16-bit** | GBs | Simple deployment; standard inference servers (vLLM, TGI) |
| **GGUF quantized** | ~1 GB for 1.5B | Local/CPU inference via Ollama or llama.cpp |

> **A note on merging quantized models.** Merging a LoRA adapter trained on a 4-bit base into 16-bit weights
> introduces a small numerical discrepancy. In practice the effect is minor, but if quality matters, evaluate the
> merged model rather than assuming it matches the adapter-based one.

In [ ]:
# Cell 11 — save the adapter (tens of MB, not gigabytes)
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

In [ ]:
# Ship a single standalone model (base + adapter fused) — merge into 16-bit weights
model.save_pretrained_merged("merged_model", tokenizer, save_method="merged_16bit")

# Or export GGUF to run locally in Ollama / llama.cpp
# model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Reloading later (in a fresh runtime)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_adapter",     # adapter dir; the base is fetched automatically
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

## §16 — Appendix A: the "vanilla" Hugging Face version

The same QLoRA setup without Unsloth, so you can see exactly what is happening underneath
(`transformers` + `peft` + `bitsandbytes` + `trl`). Run this in a **fresh runtime** — it loads a second model.

**What Unsloth adds** on top of this: fused Triton kernels for the LoRA forward/backward pass, a manual backprop
implementation that avoids redundant work, a memory-efficient gradient checkpointing variant, and pre-quantized model
weights for faster downloads. The math is identical; the difference is speed and peak memory.

In [ ]:
# Appendix A — plain Hugging Face QLoRA (run in a fresh runtime)
!pip install -q transformers datasets peft trl bitsandbytes accelerate

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
BF16 = torch.cuda.is_bf16_supported()

# 1) 4-bit quantization config — this is the "Q" in QLoRA
bnb = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",     # NormalFloat4: information-theoretically
                                           # optimal for normally-distributed weights
    bnb_4bit_compute_dtype    = torch.bfloat16 if BF16 else torch.float16,
    bnb_4bit_use_double_quant = True,      # quantize the quantization constants too
)

# 2) load base model in 4-bit + tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# casts layernorms to fp32, enables gradient checkpointing, makes inputs require grad
model = prepare_model_for_kbit_training(model)

# 3) attach LoRA adapters
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # ~1% of params trainable

# 4) train with the SAME trl.SFTTrainer + SFTConfig as in section 8.4

## §17 — Appendix B: checking the installed API yourself

This two-line habit is more durable than any rename table, including the one in the written document.

In [ ]:
from dataclasses import fields
from trl import SFTConfig
print(sorted(f.name for f in fields(SFTConfig)))   # every accepted argument

import inspect
from trl import SFTTrainer
print(inspect.signature(SFTTrainer.__init__))

## §14 — Exercises

Work at least one. Each names what to report, so results are comparable across the class.

1. **Rank sweep.** Re-run with `r = 8, 16, 32` (keep `alpha = 2r`). *Report:* final validation loss, ROUGE, adapter size on disk, wall-clock time.
2. **Learning-rate sensitivity.** Try `lr = 5e-5, 2e-4, 1e-3`. *Report:* loss curves on shared axes.
3. **Data scaling.** Train on 500 / 2,000 / 5,000 examples for a fixed 2 epochs. *Report:* train–validation gap at each size.
4. **Response-only loss.** Enable §8.6 and re-run. *Report:* five generations before and after; explain why the loss numbers are not comparable.
5. **Before vs. after.** Fill in the §11.2 rubric table on five prompts of your own.
6. **Attention-only LoRA.** `target_modules = ["q_proj","k_proj","v_proj","o_proj"]`. *Report:* trainable params, adapter size, validation loss vs. the all-linear run.
7. **Scale up.** Swap to `unsloth/Qwen2.5-7B-Instruct`. *Report:* peak VRAM and time per step vs. the 1.5B run.
8. **The negative result** *(recommended)*. Fine-tune on 50 made-up facts, then test whether the model reproduces them. *Report:* accuracy, and what it demonstrates about §3.

---

*See `Fine_Tuning_LLMs_Tutorial.md` for the full write-up: concepts, the memory arithmetic, argument reference,
pitfalls, glossary, and references.*